# TDI — DQ Check (All AUs)

## Purpose
Consolidated DQ checks for **all three TDI Assessment Units**:
- **101522** — TD General Insurance (TDGI)
- **101523** — Life & Health (L&H)
- **101524** — TD Re-insurance Barbados (TDRB)

## What Changed
The original 3 separate notebooks duplicate ~95% of their code (identical helper functions,
identical config setup, identical summary queries). This notebook:
1. Defines shared functions **once**
2. Uses a **config-driven** approach — each AU is a dictionary of `(table, column, CDEs)` tuples
3. Runs all checks in a single notebook with clear section markers

## Pattern
One check per (table, column), recording which CDEs use each column.
Results are inserted into the DQ availability table.

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Shared Functions
Defined **once** — used by all three AUs.

In [ ]:
from datetime import date

TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())


def insertDQTable(lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct):
    """Insert a DQ result row into the availability table."""
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True


def check_column(lob_id, cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        # Handle backtick-quoted column names (e.g., `Incoming-Deposits`)
        col_ref = column if column.startswith('`') else column
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({col_ref} as STRING) IS NOT NULL
                         AND trim(cast({col_ref} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(lob_id, cde_no, cde_desc, source, src_table, column, pct)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")


def run_au_checks(lob_id, lob_desc, checks):
    """
    Run all DQ checks for an AU.
    
    Args:
        lob_id: AU identifier (e.g., '101522')
        lob_desc: Human-readable AU name
        checks: list of (cde_no, cde_desc, source, table, column) tuples
    """
    print(f'\n{"=" * 70}')
    print(f'DQ Check: {lob_id} — {lob_desc}')
    print(f'Results table: {TABLE_NAME}')
    print(f'Run date: {today_date}')
    print(f'{"=" * 70}\n')
    
    current_table = None
    for cde_no, cde_desc, source, table, column in checks:
        if table != current_table:
            current_table = table
            print(f'\n{table}')
            print('-' * 60)
        check_column(lob_id, cde_no, cde_desc, source, table, column)
    
    # Display summary
    results = spark.sql(f"""
        SELECT * FROM {TABLE_NAME}
        WHERE lob_id = '{lob_id}'
          AND run_date = '{today_date}'
        ORDER BY src_table_name, data_element
    """)
    print(f'\nTotal DQ checks: {results.count()}')
    display(results)


print('Shared functions defined.')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

---
## AU: 101522 — TD General Insurance (TDGI)

### Source Tables
| Table | CDEs |
|-------|------|
| `ra_fy_2025.joyce_tdigi_2725` | SD1, 1.7, 4.1A, SD3, 1.6 |
| `ra_fy_2025.joyce_tdigi_2727` | SD4 |
| `ra_fy_2025.joyce_tdigi_3723` | 6.2B |
| `ra_fy_2025.joyce_tdigi_8729` | 4.1B |
| `ra_adido_2025.tdigi_legal_structure_2025` | SD5 |
| `RA_FY24_VIEW.tdgi_active_policies` | 6.5A |
| `RA_FY25_VIEW.TD_Insurance_Branches_2025` | 5.1–5.9 |
| + 16 shared reference/centralized tables |

In [ ]:
# ============================================================
# 101522 — TDGI check definitions
# Each tuple: (cde_no, cde_desc, source, table, column)
# ============================================================
TDGI_CHECKS = [
    # --- Segment: joyce_tdigi_2725 ---
    ('SD1,1.7,4.1A', 'Segment/Centralized', 'Rahona', 'ra_fy_2025.joyce_tdigi_2725', 'acc_account_no'),
    ('SD3,1.6,1.7', 'Segment/Centralized', 'Rahona', 'ra_fy_2025.joyce_tdigi_2725', 'accpn_pholder_country_de'),
    # --- Segment: joyce_tdigi_2727 ---
    ('SD4', 'Segment', 'Rahona', 'ra_fy_2025.joyce_tdigi_2727', 'accpn_pholder_occupation_tx'),
    # --- Segment: joyce_tdigi_3723 ---
    ('6.2B', 'Segment', 'Rahona', 'ra_fy_2025.joyce_tdigi_3723', 'acc_account_no'),
    # --- Segment: joyce_tdigi_8729 ---
    ('4.1B', 'Segment', 'Rahona', 'ra_fy_2025.joyce_tdigi_8729', 'acc_account_no'),
    # --- Segment: tdigi_legal_structure ---
    ('SD5', 'Segment', 'ADIDO', 'ra_adido_2025.tdigi_legal_structure_2025', 'Legal_Structure'),
    ('SD5', 'Segment', 'ADIDO', 'ra_adido_2025.tdigi_legal_structure_2025', 'Accounts'),
    # --- Segment: active policies (6.5A) ---
    ('6.5A', 'Segment', 'Rahona', 'RA_FY24_VIEW.tdgi_active_policies', 'acc_account_no'),
    ('6.5A', 'Segment', 'Rahona', 'RA_FY24_VIEW.tdgi_active_policies', 'tdihps_account_month_end_in'),
    ('6.5A', 'Segment', 'Rahona', 'RA_FY24_VIEW.tdgi_active_policies', 'tdihps_asof_dt'),
    # --- Reference: sanctions country ---
    ('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO', 'ra_adido_2025.sanctions_country_risk_rating_2025', 'country'),
    ('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO', 'ra_adido_2025.sanctions_country_risk_rating_2025', 'riskrating'),
    # --- Reference: occupation ---
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.occupation_code_list_Ca2025', 'Occupation_Name'),
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.occupation_code_list_Ca2025', 'Updated_Risk_Rating'),
    # --- Reference: industry ---
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Code'),
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Description'),
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.industry_ref_list_ca2025', 'Updated_Risk_Rating'),
    # --- Reference: entity ---
    ('SD5', 'Reference', 'ADIDO', 'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Type_Name'),
    ('SD5', 'Reference', 'ADIDO', 'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Category'),
    ('SD5', 'Reference', 'ADIDO', 'ra_adido_2025.entity_ref_list_ca2025', 'Updated_Risk_Ratings'),
    # --- Branch ---
    ('5.1,5.9', 'Branch', 'ADIDO', 'RA_FY25_VIEW.TD_Insurance_Branches_2025', 'general_ins_ind'),
    ('5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Branch', 'ADIDO', 'RA_FY25_VIEW.TD_Insurance_Branches_2025', 'country'),
    # --- Policy history (1.7) ---
    ('1.7', 'Centralized', 'Rahona', 'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'acc_account_no'),
    ('1.7', 'Centralized', 'Rahona', 'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'pol_valid_la'),
    ('1.7', 'Centralized', 'Rahona', 'ra_fy_2025.prod30tdi_tddi_to_active_policies_history', 'pol_effective_dt'),
    ('1.7', 'Centralized', 'Rahona', 'ra_fy_2025.prod30tdi_tdi_account_person', 'acc_account_no'),
    ('1.7', 'Centralized', 'Rahona', 'ra_fy_2025.prod30tdi_tdi_account_person', 'accpn_pholder_country_de'),
    # --- Centralized / Sanctions ---
    ('1.8', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanctions_match_2025', 'Customername'),
    ('1.8', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanctions_match_2025', 'tds'),
    ('1.9', 'Centralized', 'ADIDO', 'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'ACCOUNTBLOCKED'),
    ('1.9', 'Centralized', 'ADIDO', 'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED'),
    # --- CRR ---
    ('1.1', 'Centralized', 'Rahona', 'rafy2025_centralized.crv_scorable_cust_ca', 'CUST_INTRL_ID'),
    ('1.1', 'Centralized', 'Rahona', 'rafy2025_centralized.crv_scorable_cust_ca', 'v_entity_id'),
    ('1.2,1.3,1.4', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_rated_ca', 'CUST_INTRL_ID'),
    ('1.2,1.3,1.4', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_rated_ca', 'risk_rating'),
    ('1.5', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_unrated_ca', 'CUST_INTRL_ID'),
    ('SD6', 'Centralized', 'Rahona', 'rafy2025_centralized.CDE_SD_6_1yr_fy2025', 'full_nm'),
    # --- UTR / STR ---
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_nm'),
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no'),
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt'),
    ('3.18', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_Internal_Unique_ID'),
    ('3.18', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_LEILAs_Customer_Nam'),
    # --- ABAC ---
    ('ABAC1.2', 'ABAC', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'ENTITY'),
    ('ABAC1.2', 'ABAC', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'CUST_INTRL_ID'),
    ('ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'DerivedDesc'),
    ('ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating'),
]

run_au_checks('101522', 'TDI - General Insurance', TDGI_CHECKS)

---
## AU: 101523 — Life & Health (L&H)

### Source Tables
| Table | CDEs |
|-------|------|
| `p_parsed_2025_new.marketing_lh_customers_combined` | SD1, SD3, 1.x, 3.17–3.18, 6.5B |
| `p_parsed_2025_new.cp_new_enrollment_dates` | 4.1A, 4.1B |
| `p_parsed_2025_new.bpi_new_enrollment_dates` | 4.1A, 4.1B |
| `ra_fy25_view.tdi_rpt_main3_history_view` | 3.1, 3.6, 3.9, 3.16 |
| + shared reference/centralized tables |

In [ ]:
# ============================================================
# 101523 — L&H check definitions
# ============================================================
LH_CHECKS = [
    # --- Segment: marketing_lh_customers_combined ---
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'first_name'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'last_name'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'dob'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'customer_number'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'policy_number'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'month_end'),
    ('SD1,SD3,1.1-1.9,1.6,1.7,SD6,3.17,3.18,6.5B,ABAC1.2', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'datasource'),
    ('SD3,1.6,1.7', 'Segment', 'Parsed',
     'p_parsed_2025_new.marketing_lh_customers_combined', 'country'),
    # --- Enrollment: cp ---
    ('4.1A,4.1B', 'Segment', 'Parsed', 'p_parsed_2025_new.cp_new_enrollment_dates', 'policy_number'),
    # --- Enrollment: bpi ---
    ('4.1A,4.1B', 'Segment', 'Parsed', 'p_parsed_2025_new.bpi_new_enrollment_dates', 'policy_number'),
    # --- Reference: sanctions (all) ---
    ('1.6,1.7', 'Reference', 'Parsed', 'p_parsed_2025_new.sanctions_country_risk_rating_2025_all', 'country'),
    ('1.6,1.7', 'Reference', 'Parsed', 'p_parsed_2025_new.sanctions_country_risk_rating_2025_all', 'RiskRating'),
    # --- Reference: sanctions ---
    ('5.6,5.7', 'Reference', 'ADIDO', 'ra_adido_2025.sanctions_country_risk_rating_2025', 'country'),
    ('5.6,5.7', 'Reference', 'ADIDO', 'ra_adido_2025.sanctions_country_risk_rating_2025', 'RiskRating'),
    # --- Reference: country ---
    ('5.2,5.3,5.4,5.5', 'Reference', 'ADIDO', 'ra_adido_2025.country_ref_list_ca2025', 'Country_Name'),
    ('5.2,5.3,5.4,5.5', 'Reference', 'ADIDO', 'ra_adido_2025.country_ref_list_ca2025', '2025_Risk_Rating'),
    # --- Branch ---
    ('5.1,5.9', 'Branch', 'ADIDO', 'ra_fy25_view.td_insurance_branches_2025', 'life_health_ins_ind'),
    ('5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Branch', 'ADIDO', 'ra_fy25_view.td_insurance_branches_2025', 'country'),
    # --- L&H Custom (6.5A) ---
    ('6.5A', 'Segment', 'Rahona', 'ra_fy24_view.tdlh_custom_metrics_2024', 'lch_volume_cnt'),
    ('6.5A', 'Segment', 'Rahona', 'ra_fy24_view.tdlh_custom_metrics_2024', 'lob_channel_desc'),
    ('6.5A', 'Segment', 'Rahona', 'ra_fy24_view.tdlh_custom_metrics_2024', 'lob_type_desc'),
    # --- Transactions ---
    ('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona', 'ra_fy25_view.tdi_rpt_main3_history_view', 'postdate'),
    ('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona', 'ra_fy25_view.tdi_rpt_main3_history_view', 'endpt'),
    ('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona', 'ra_fy25_view.tdi_rpt_main3_history_view', 'curtype'),
    ('3.1,3.6,3.9,3.16', 'Transaction', 'Rahona', 'ra_fy25_view.tdi_rpt_main3_history_view', 'curtime_amount'),
    # --- Centralized / Sanctions ---
    ('1.8', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanction_match_2025', 'Customername'),
    ('1.9', 'Centralized', 'ADIDO', 'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED'),
    # --- CRR ---
    ('1.1', 'Centralized', 'Rahona', 'rafy2025_centralized.crv_scorable_cust_ca', 'CUST_INTRL_ID'),
    ('1.1', 'Centralized', 'Rahona', 'rafy2025_centralized.crv_scorable_cust_ca', 'v_entity_id'),
    ('1.2,1.3,1.4', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_rated_ca', 'CUST_INTRL_ID'),
    ('1.2,1.3,1.4', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_rated_ca', 'risk_rating'),
    ('1.5', 'Centralized', 'Rahona', 'rafy2025_centralized.customer_scorable_unrated_ca', 'CUST_INTRL_ID'),
    ('SD6', 'Centralized', 'Rahona', 'rafy2025_centralized.CDE_SD_6_1yr_fy2025', 'full_nm'),
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_nm'),
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no'),
    ('3.17', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt'),
    ('3.18', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_Internal_Unique_ID'),
    ('3.18', 'Centralized', 'Rahona', 'rafy2025_centralized.TD_SAR_CDE_3_18_2025', 'STR_LEILAs_Customer_Nam'),
    # --- ABAC ---
    ('ABAC1.2', 'ABAC', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'ENTITY'),
    ('ABAC1.2', 'ABAC', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'CUST_INTRL_ID'),
    ('ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'DerivedDesc'),
    ('ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating'),
]

run_au_checks('101523', 'TDI - Life & Health', LH_CHECKS)

---
## AU: 101524 — TD Re-insurance Barbados (TDRB)

### Source Tables
| Table | CDEs |
|-------|------|
| `ra_adido_2025.101524_tdi_barbados_customer` | SD1, SD3–SD6, 1.x, 3.17–3.18, 4.1x, 6.5B |
| `ra_adido_2025.barbados_registered_office` | 5.1–5.8 |
| `ra_adido_2025.barbados_transaction_fy25` | SD7, SD8, 3.1–3.16 |
| `ra_adido_2025.barbados_centralized_fy25` | 1.1–1.5, SD2 |
| `ra_adido_2025.barbados_str` | 3.17, 3.18 |
| + shared reference tables |

In [ ]:
# ============================================================
# 101524 — TDRB check definitions
# ============================================================
TDRB_CHECKS = [
    # --- Segment: barbados_customer ---
    ('SD1,SD3,SD4,SD5,1.6,1.7,1.8,1.9,3.17,3.18,4.1A,4.1B,4.1C,6.5B,SD6,ABAC1.1,ABAC1.3,SD1.1,SD1.3',
     'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Unique_ID'),
    ('SD3,1.6', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'ISOA_Country_Code'),
    ('SD4', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'SIC_Code'),
    ('SD4', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Standardized_SIC_Code'),
    ('SD5', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Legal_Entity_Classification'),
    ('SD6', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Length_of_Relationship'),
    ('6.5B', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Status'),
    ('4.1A', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'DeliveryChannel'),
    ('1.8,1.9', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Name_Beneficial_Owner'),
    ('3.17,3.18', 'Segment', 'ADIDO', 'ra_adido_2025.101524_tdi_barbados_customer', 'Counterparty_Name'),
    # --- Branch: barbados_registered_office ---
    ('5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,ABAC5.1', 'Branch', 'ADIDO', 'ra_adido_2025.barbados_registered_office', 'address'),
    ('5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,ABAC5.1', 'Branch', 'ADIDO', 'ra_adido_2025.barbados_registered_office', 'country'),
    ('5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,ABAC5.1', 'Branch', 'ADIDO', 'ra_adido_2025.barbados_registered_office', 'office'),
    # --- Transaction: barbados_transaction ---
    ('SD7,SD8,3.1-3.16,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO', 'ra_adido_2025.barbados_transaction_fy25', 'FileName'),
    ('SD7,SD8,3.1-3.16,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO', 'ra_adido_2025.barbados_transaction_fy25', 'Jurisdiction'),
    ('SD7,SD8,3.1-3.16,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO', 'ra_adido_2025.barbados_transaction_fy25', '`Incoming-Deposits`'),
    ('SD7,SD8,3.1-3.16,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO', 'ra_adido_2025.barbados_transaction_fy25', '`Outgoing-Payments`'),
    # --- Centralized: barbados_centralized ---
    ('1.1,1.2,1.3,1.4,1.5,SD2,ABAC1.2', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_centralized_fy25', 'Customer_Unique_ID'),
    ('1.1,1.2,1.3,1.4,1.5', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_centralized_fy25', 'TDRISKSRATING'),
    ('SD2,ABAC1.2', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_centralized_fy25', '`POLITICALLYEXPOSEDPERSON/HDIassociates`'),
    # --- Sanctions ---
    ('1.8', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanction_match_2025', 'Customername'),
    ('1.8', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanction_match_2025', 'CustomerType'),
    ('1.9', 'Centralized', 'ADIDO', 'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED'),
    # --- STR ---
    ('3.17,3.18', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_str', 'details'),
    ('3.17,3.18', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_str', 'date'),
    ('3.17,3.18', 'Centralized', 'ADIDO', 'ra_adido_2025.barbados_str', 'amount'),
    # --- Reference: industry ---
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Code'),
    ('SD4', 'Reference', 'ADIDO', 'ra_adido_2025.industry_ref_list_ca2025', 'Updated_Risk_Rating'),
    # --- Reference: entity ---
    ('SD5', 'Reference', 'ADIDO', 'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Type_Name'),
    ('SD5', 'Reference', 'ADIDO', 'ra_adido_2025.entity_ref_list_ca2025', 'Updated_Risk_Ratings'),
    # --- Reference: country ---
    ('5.2,5.3,5.4,5.5,3.1-3.4,3.9-3.12,ABAC1.1', 'Reference', 'ADIDO', 'ra_adido_2025.country_ref_list_ca2025', 'Country_Name'),
    ('5.2,5.3,5.4,5.5,3.1-3.4,3.9-3.12,ABAC1.1', 'Reference', 'ADIDO', 'ra_adido_2025.country_ref_list_ca2025', '`2025_Risk_Rating`'),
    ('ABAC1.1', 'Reference', 'ADIDO', 'ra_adido_2025.country_ref_list_ca2025', 'ISOA_Country_Code'),
    # --- Reference: sanctions ---
    ('1.6,5.5-5.8,3.5-3.8,3.13-3.16', 'Reference', 'ADIDO', 'ra_fy25_view.sanctions_country_risk_rating_fy2025', 'Country_Name'),
    ('1.6,5.5-5.8,3.5-3.8,3.13-3.16', 'Reference', 'ADIDO', 'ra_fy25_view.sanctions_country_risk_rating_fy2025', 'RiskRating'),
    # --- ABAC ---
    ('ABAC1.1,ABAC3.1,ABAC3.2,ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating'),
    ('ABAC1.1,ABAC3.1,ABAC3.2,ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'Jurisdiction'),
    ('ABAC1.1,ABAC3.1,ABAC3.2,ABAC5.1', 'ABAC', 'ADIDO', 'ra_adido_2025.TD_Country_Risk_Rating_ABAC', '`Alpha-3Code`'),
    ('ABAC1.3', 'ABAC', 'ADIDO', 'ra_adido_2025.Industry_Reference_List_08222025_df', 'IndustryCode'),
    ('ABAC1.3', 'ABAC', 'ADIDO', 'ra_adido_2025.Industry_Reference_List_08222025_df', 'ABACRse'),
    # --- Historical results (6.5A) ---
    ('6.5A', 'Historical', 'ADIDO', 'ra_adhor_data.101524_cde_results_fy2024', 'values'),
    ('6.5A', 'Historical', 'ADIDO', 'ra_adhor_data.101524_cde_results_fy2024', 'CDE'),
]

run_au_checks('101524', 'TDI - TD Re-insurance Barbados', TDRB_CHECKS)

---
## Cross-AU Summary
Compare results across all three TDI Assessment Units.

In [ ]:
# ============================================================
# Cross-AU summary — all TDI AUs in one view
# ============================================================
results = spark.sql(f"""
    SELECT lob_id, 
           count(1) as total_checks,
           round(avg(cast(availability_pct as double)), 2) as avg_availability,
           min(cast(availability_pct as double)) as min_availability,
           count(CASE WHEN cast(availability_pct as double) < 100 THEN 1 END) as below_100
    FROM {TABLE_NAME}
    WHERE lob_id IN ('101522', '101523', '101524')
      AND run_date = '{today_date}'
    GROUP BY lob_id
    ORDER BY lob_id
""")
display(results)

# Detail: any columns below 100% availability
issues = spark.sql(f"""
    SELECT lob_id, cde_no, src_table_name, data_element, availability_pct
    FROM {TABLE_NAME}
    WHERE lob_id IN ('101522', '101523', '101524')
      AND run_date = '{today_date}'
      AND cast(availability_pct as double) < 100
    ORDER BY lob_id, availability_pct
""")
if issues.count() > 0:
    print(f'\nColumns below 100% availability: {issues.count()}')
    display(issues)
else:
    print('\nAll columns at 100% availability!')

---
## Notes

### Sanctions Table Name Inconsistency
> **TDGI** uses `true_sanction**s**_match_2025` (plural 's')  
> **L&H / TDRB** use `true_sanction_match_2025` (singular)  
> Verify which is the correct table name.

### Column Differences
| Check | TDGI | L&H | TDRB |
|-------|------|-----|------|
| Sanctions columns | `Customername` + `tds` | `Customername` only | `Customername` + `CustomerType` |
| Blocked property | `ACCOUNTBLOCKED` + `CUSTOMERIDIMPACTED` | `CUSTOMERIDIMPACTED` only | `CUSTOMERIDIMPACTED` only |
| Branch indicator | `general_ins_ind` | `life_health_ins_ind` | N/A (uses `office`) |

### Original Notebooks (Preserved)
- `TDI_101522_TDGI_DQ_Check.ipynb`
- `TDI_101523_LH_DQ_Check.ipynb`
- `TDI_101524_TDRB_DQ_Check.ipynb`